# MOE Medical Vision - Download Datasets

Ejecutar celda por celda en orden.

In [2]:
import os
import subprocess
import zipfile
from pathlib import Path

RAW_DIR = Path("/workspace/moe_medical_vision/data/raw")

os.environ['KAGGLE_USERNAME'] = 'alej0909'
os.environ['KAGGLE_KEY'] = 'KGAT_55c43d8a96170bc499971e7337c36a50'

print('Setup listo!')

Setup listo!


## 1. NIH ChestX-ray14 (~45GB)

In [4]:
# Descargar NIH con Kaggle CLI
import subprocess
import threading
import time

def monitor_disk(stop_event):
    while not stop_event.is_set():
        result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
        lines = result.stdout.strip().split('\n')
        if len(lines) > 1:
            parts = lines[1].split()
            print(f'\rEspacio usado: {parts[4]} | Disponibles: {parts[3]}', end='', flush=True)
        time.sleep(5)

print('Descargando NIH (~45GB) con Kaggle CLI...')
print('Monitor de disco:')

stop_event = threading.Event()
monitor_thread = threading.Thread(target=monitor_disk, args=(stop_event,))
monitor_thread.start()

try:
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'nih-chest-xrays/data', '-p', str(NIH_DIR), '--unzip'],
        capture_output=True, text=True, timeout=7200
    )
finally:
    stop_event.set()
    monitor_thread.join()

print()  # newline after monitor

if result.returncode == 0:
    print('OK: NIH descargado!')
    
    # Limpiar cache para liberar espacio
    subprocess.run(['rm', '-rf', '/root/.cache'], capture_output=True)
    
    # Verificar
    size_result = subprocess.run(['du', '-sh', str(NIH_DIR)], capture_output=True, text=True)
    print(f'Tamano final: {size_result.stdout.strip()}')
else:
    print(f'ERROR: {result.stderr[-500:]}')
    print('Reintentando...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'nih-chest-xrays/data', '-p', str(NIH_DIR), '--unzip'],
        capture_output=True, text=True, timeout=7200
    )
    if result.returncode == 0:
        print('OK en retry!')
    else:
        print(f'ERROR final: {result.stderr[-500:]}')

Descargando NIH (~45GB) con Kaggle CLI...
Monitor de disco:
Espacio usado: 18% | Disponibles: 25G
OK: NIH descargado!
Tamano final: 43G	/workspace/moe_medical_vision/data/raw/nih


## 2. ISIC 2019 (~5GB)

In [5]:
print('Descargando ISIC 2019 (~5GB)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'andrewmvd/isic-2019', '-p', str(RAW_DIR / 'isic'), '--unzip'],
    capture_output=True, text=True, timeout=7200
)
if result.returncode == 0:
    print('OK: ISIC 2019 descargado!')
else:
    print(f'ERROR: {result.stderr[-500:]}')

Descargando ISIC 2019 (~5GB)...
OK: ISIC 2019 descargado!


## 3. Osteoarthritis Knee (~10GB)

In [4]:
print('Descargando Osteoarthritis (~10GB)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'dhruvacube/osteoarthritis', '-p', str(RAW_DIR / 'osteoporosis'), '--unzip'],
    capture_output=True, text=True, timeout=7200
)
if result.returncode == 0:
    print('OK: Osteoarthritis descargado!')
else:
    print(f'ERROR: {result.stderr[-500:]}')

Descargando Osteoarthritis (~10GB)...
OK: Osteoarthritis descargado!


## 4. LUNA16 (~1GB)

In [5]:
print('Descargando LUNA16 (~1GB)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'fanbyprinciple/luna-lung-cancer-dataset', '-p', str(RAW_DIR / 'luna16'), '--unzip'],
    capture_output=True, text=True, timeout=7200
)
if result.returncode == 0:
    print('OK: LUNA16 descargado!')
else:
    print(f'ERROR: {result.stderr[-500:]}')

Descargando LUNA16 (~1GB)...
OK: LUNA16 descargado!


## 5. Pancreatic Cancer (~46GB - Zenodo)

In [6]:
print('Descargando Pancreatic Cancer desde Zenodo (~46GB)...')
dest = RAW_DIR / 'pancreatic'
zip_file = dest / 'batch_1.zip'

result = subprocess.run(
    ['wget', '--progress=bar:force', '-O', str(zip_file), 
     'https://zenodo.org/records/13715870/files/batch_1.zip'],
    timeout=28800
)

if result.returncode == 0 and zip_file.exists():
    print('Descarga completa. Extrayendo...')
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(dest)
    print('OK: Pancreatic Cancer extraido!')
    print('Eliminando zip para liberar espacio...')
    zip_file.unlink()
    print('OK: Espacio liberado')
else:
    print('ERROR: Fallo en descarga')

Descargando Pancreatic Cancer desde Zenodo (~46GB)...


--2026-03-25 18:18:28--  https://zenodo.org/records/13715870/files/batch_1.zip
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49338585294 (46G) [application/octet-stream]
Saving to: ‘/workspace/moe_medical_vision/data/raw/pancreatic/batch_1.zip’

/workspace/moe_medi 100%[===================>]  45.95G  32.4MB/s    in 38m 5s  

2026-03-25 18:56:34 (20.6 MB/s) - ‘/workspace/moe_medical_vision/data/raw/pancreatic/batch_1.zip’ saved [49338585294/49338585294]



Descarga completa. Extrayendo...
OK: Pancreatic Cancer extraido!
Eliminando zip para liberar espacio...
OK: Espacio liberado


## 6. Reporte de Integridad

In [ ]:
def get_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total

def count_files(path):
    return sum(len(files) for _, _, files in os.walk(path))

print('='*60)
print(' REPORTE DE INTEGRIDAD')
print('='*60)

datasets = ['nih', 'isic', 'osteoporosis', 'luna16', 'pancreatic']
total = 0

for ds in datasets:
    folder = RAW_DIR / ds
    if folder.exists():
        n = count_files(folder)
        s = get_size(folder) / (1024**3)
        total += s
        print(f'{ds:<15} {n:>8} archivos  {s:>8.2f} GB')
    else:
        print(f'{ds:<15} {'0':>8} archivos  {'0.00':>8} GB')

print('='*60)
print(f'{'TOTAL':<15} {'':>8}          {total:>8.2f} GB')
print('='*60)